In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
# --------------------------
# 基础模块 (CBR, Head, DownSample)
# --------------------------

class CBR(nn.Sequential):
    def __init__(self, ins, outs, k, s, p, d=1):
        super(CBR, self).__init__(
            nn.Conv2d(in_channels=ins, out_channels=outs, kernel_size=k, stride=s, padding=p, dilation=d, bias=False),
            nn.BatchNorm2d(num_features=outs),
            nn.ReLU(inplace=True)
        )

class DownSample(nn.Module):
    def __init__(self, ins, outs, k, s):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=ins, out_channels=outs, kernel_size=k, stride=s, bias=False),
            nn.BatchNorm2d(num_features=outs)
        )
    def forward(self, x):
        return self.block(x)


# --------------------------
# 核心 Bottleneck
# --------------------------

class Bottleneck(nn.Module):
    def __init__(self, ins, outs, k, s, p, d, downsample=None):
        super().__init__()
        self.ds = downsample
        # 1x1 降维
        self.cbr1 = CBR(ins, outs, 1, 1, 0)
        # 3x3 处理 (注意这里的 dilation)
        self.cbr2 = CBR(outs, outs, 3, s, p, d)
        # 1x1 升维 (Expansion=4)
        self.cv = nn.Conv2d(outs, outs * 4, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(outs * 4)

    def forward(self, x):
        identity = x
        out = self.cbr1(x)
        out = self.cbr2(out)
        out = self.cv(out)
        out = self.bn(out)
        
        if self.ds is not None:
            identity = self.ds(x)
            
        out += identity
        return F.relu(out, inplace=True)

# --------------------------
# 各个 Layer 实现
# --------------------------

class Layer_1(nn.Module):
    def __init__(self, ins, outs, k=3, s=1, p=1, d=1):
        super().__init__()
        self.expansion = 4
        # Layer 1 输入通常与输出基数一致(64->64)，但Expansion后变256，所以需要Downsample
        ds = DownSample(ins, outs * self.expansion, 1, s)
        
        self.b1 = Bottleneck(ins, outs, k, s, p, d, downsample=ds)
        self.b2 = Bottleneck(outs * self.expansion, outs, k, 1, p, d)
        self.b3 = Bottleneck(outs * self.expansion, outs, k, 1, p, d)

    def forward(self, x):
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        return x

class Layer_2(nn.Module):
    def __init__(self, ins, outs): 
        # Ins: 256, Outs(Base): 128 -> Final: 512
        super().__init__()
        exp = 4
        self.b = nn.Sequential(
            # Stride=2 进行下采样
            Bottleneck(ins, outs, 3, 2, 1, 1, downsample=DownSample(ins, outs * exp, 1, 2)),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1)
        )
    def forward(self, x):
        return self.b(x)

class Layer_3(nn.Module):
    def __init__(self, ins, outs):
        super().__init__()
        exp = 4
        
        self.b = nn.Sequential(
            Bottleneck(ins, outs, 3, 2, 1, 1, downsample=DownSample(ins, outs * exp, 1, 2)),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1), 
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
        )
    def forward(self, x):
        return self.b(x)

class Layer_4(nn.Module):
    def __init__(self, ins, outs):
        super().__init__()
        exp = 4
        self.b = nn.Sequential(
            Bottleneck(ins, outs, 3, 2, 1, 1, downsample=DownSample(ins, outs * exp, 1, 2)),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1), 
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
        )
    def forward(self, x):
        return self.b(x)

# --------------------------


class Encoder(nn.Module):
    def __init__(self, ins=3):
        super().__init__()
        
        # 1. Stem (前处理)
        # 输入: [B, 3, 480, 480] -> 输出: [B, 64, 120, 120]
        # cbr(ins, outs, k, s, p, d=1):
        
        self.cbr=CBR(ins, 64, 7, 2, 3)  # -> /2
            #k=3,s=2,p=1
        self.m=nn.MaxPool2d(3, 2, 1)   # -> /4
        
        
        # 2. Backbone (骨干)
        self.layer1 = Layer_1(ins=64, outs=64)
        
    
        self.layer2 = Layer_2(ins=256, outs=128)
        
        self.layer3 = Layer_3(ins=512, outs=256)
     
        self.layer4 = Layer_4(ins=1024, outs=512)
        
     

    def forward(self, x):
        input_size = x.shape[2:] # 记录 H, W: (224, 224)
        
        # --- 编码阶段 (Encoder) ---
        x = self.cbr(x)
        f1=x#跳跃链接
        x=self.m(x)
        x = self.layer1(x)
        f2=x#跳跃链接
        x = self.layer2(x)
        f3=x#跳跃链接
        x = self.layer3(x)
        f4=x#跳跃链接
        x = self.layer4(x)#Layer 4 Output: torch.Size([B, 2048, 16, 16])
        
        
        
        return x,[f1,f2,f3,f4]
    
    
class CCR(nn.Module):
    def __init__(self, ins, outs, k=3, s=1, p=1): # 注意 p=1 配合 k=3 才能保持尺寸
        super().__init__()
        self.b = nn.Sequential(
            # 第一层：负责把通道数降下来 (例如 3072 -> 512)
            nn.Conv2d(ins, outs, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(outs), # 强烈建议加 BN
            nn.ReLU(inplace=True),
            
            # 第二层：输入已经是 outs 了！
            nn.Conv2d(outs, outs, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(outs),
            nn.ReLU(inplace=True)
        ) 
    def forward(self, x):
        return self.b(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        """
        in_ch:   来自上一层(深层)的通道数 (例如 2048)
        skip_ch: 来自Encoder跳跃连接的通道数 (例如 1024)
        out_ch:  输出的目标通道数 (例如 512)
        """
        super().__init__()
        # 1. 上采样
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        # 2. 计算拼接后的总通道数 (自动计算，不用写死!)
        # 拼接后通道 = 上一层通道(in_ch) + 跳跃连接通道(skip_ch)
        concat_ch = in_ch + skip_ch 
        
        # 3. 定义卷积层
        self.ccr = CCR(ins=concat_ch, outs=out_ch)
        
    def forward(self, x, skip):
        # x: 来自深层
        # skip: 来自跳跃连接
        
        x = self.up(x)#上采样
        
        # 维度检查 (可选，防止尺寸由于取整问题不匹配)
        if x.size(2) != skip.size(2):
            x = F.interpolate(x, size=skip.shape[2:])
            
        x = torch.cat([skip, x], dim=1) # 在通道维拼接
        x = self.ccr(x)
        return x


class Decoder(nn.Module):
    def __init__(self, nums=64,filters=[64, 256, 512, 1024], bottom_ch=2048):
        """
        filters: Encoder 各层跳跃连接的通道数列表 [Layer1, Layer2, Layer3, Layer4_Skip]
                 对应 ResNet50: [64(Stem), 256, 512, 1024]
        bottom_ch: 最底层(Layer4 Output)的通道数，ResNet50 是 2048
        """
        super().__init__()
        # filters = [64, 256, 512, 1024]
        # 对应的 skip 顺序是从深到浅: f4(1024) -> f3(512) -> f2(256) -> f1(64)
        
        # Block 4: 融合 Bottom(2048) + Skip4(1024)
        # Out 设置为 filters[2] (512) 以便对接下一层
        self.dec4 = DecoderBlock(in_ch=bottom_ch, skip_ch=filters[3], out_ch=filters[2])
        
        # Block 3: 融合 Result(512) + Skip3(512)
        # Out 设置为 filters[1] (256)
        self.dec3 = DecoderBlock(in_ch=filters[2], skip_ch=filters[2], out_ch=filters[1])
        
        # Block 2: 融合 Result(256) + Skip2(256)
        # Out 设置为 128 (开始逐渐减小通道)
        self.dec2 = DecoderBlock(in_ch=filters[1], skip_ch=filters[1], out_ch=128)
        
        # Block 1: 融合 Result(128) + Skip1(64, Stem输出)
        # Out 设置为 64
        self.dec1 = DecoderBlock(in_ch=128, skip_ch=filters[0], out_ch=64)
        self.head = nn.Conv2d(filters[0],nums,kernel_size=1)
    def forward(self, x, skips):
        """
        x: Encoder 最后一层的输出 [B, 2048, H/32, W/32]
        skips: 跳跃连接列表 [f_stem, f_layer1, f_layer2, f_layer3]
               对应通道: [64, 256, 512, 1024]
               这里的是向量列表
        """
        # 1. 拆解 skips (注意顺序，f4是最深的，f1是最浅的)
        # 假设 skips 传进来是 [Stem, L1, L2, L3]
        f1, f2, f3, f4 = skips 
        
        # 2. 从下往上解码
        x = self.dec4(x, f4) # 融合 Layer4Out + Layer3
        x = self.dec3(x, f3) # 融合 + Layer2
        x = self.dec2(x, f2) # 融合 + Layer1
        x = self.dec1(x, f1) # 融合 + Stem
        #3.再次经过
        x=self.head(x)
        return x

class UNet(nn.Module):
    def __init__(self,ins=3,nums=64):
        super().__init__()
        self.ec = Encoder(ins)
        self.dc= Decoder(nums)
    def forward(self,x):
        #x 这里是原始尺寸[B 3 512 512]
        input_size = x.shape[2:]
        x,f = self.ec(x)
        x=self.dc(x,f)
        if x.size(2) != input_size[0] or x.size(3) != input_size[1]:
            x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=True)
            
        return x

In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(3,64).to(device)
dummy_input = torch.randn(2, 3, 512, 512).to(device)

print("Testing UNet final version...")
output = model(dummy_input)

print(f"Input: {dummy_input.shape}")
print(f"Output: {output.shape}")

# 验证是否成功
assert output.shape == (2, 64, 512, 512), "Output shape mismatch!"
print("✅ Model passed shape check!")

Testing UNet final version...
Input: torch.Size([2, 3, 512, 512])
Output: torch.Size([2, 64, 512, 512])
✅ Model passed shape check!


In [30]:


dummy_input = torch.randn(2, 3, 512, 512)
model = Encoder(ins=3)

print("正在测试模型流向...")
try:
    output ,f= model(dummy_input)
    print("\n✅ 模型运行成功!")
    print(f"输入尺寸: {dummy_input.shape}")
    print(f"输出尺寸: {output.shape}")
    
    # 验证 Layer 4 
    # 224 / 16 = 14
    stem_out = model.cbr(dummy_input)
    stem_out=model.m(stem_out)
    l1_out = model.layer1(stem_out)
    l2_out = model.layer2(l1_out)
    l3_out = model.layer3(l2_out)
    l4_out = model.layer4(l3_out)
    
    print("\n--- 内部尺寸检查 ---")
    print(f"Layer 1 Output: {l1_out.shape} ")
    print(f"Layer 2 Output: {l2_out.shape} ")
    print(f"Layer 3 Output: {l3_out.shape} ")
    print(f"Layer 4 Output: {l4_out.shape} ")
    
except Exception as e:
    print("\n❌ 发生错误:")
    print(e)

正在测试模型流向...

✅ 模型运行成功!
输入尺寸: torch.Size([2, 3, 512, 512])
输出尺寸: torch.Size([2, 2048, 16, 16])

--- 内部尺寸检查 ---
Layer 1 Output: torch.Size([2, 256, 128, 128]) 
Layer 2 Output: torch.Size([2, 512, 64, 64]) 
Layer 3 Output: torch.Size([2, 1024, 32, 32]) 
Layer 4 Output: torch.Size([2, 2048, 16, 16]) 


In [37]:
# 3. Head (分类头)
print(l4_out.size(2))
l4_out = F.interpolate(l4_out, size=l4_out.size(2)*2, mode='bilinear', align_corners=True)
print(l4_out.shape)
filters=[]
for x in f:
    filters.append(x.size(1))
for x in filters:
    print(x)

64
torch.Size([2, 2048, 128, 128])
64
256
512
1024
